In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


In [2]:
RAW_PATH = "../data/dirty_cafe_sales.csv"
df = pd.read_csv(RAW_PATH)

df.head()


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
print("Filas/Columnas:", df.shape)
df.info()
df.describe(include="all").T


Filas/Columnas: (10000, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


,count,unique,top,freq
Transaction ID,10000,10000,TXN_1961373,1
Item,9667,10,Juice,1171
Quantity,9862,7,5,2013
Price Per Unit,9821,8,3.0,2429
Total Spent,9827,19,6.0,979
Payment Method,7421,5,Digital Wallet,2291
Location,6735,4,Takeaway,3022
Transaction Date,9841,367,UNKNOWN,159


In [4]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)
df.columns


Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent', 'payment_method', 'location', 'transaction_date'], dtype='object')

In [5]:
dup_count = df.duplicated().sum()
print("Duplicados exactos:", dup_count)

if dup_count > 0:
    df = df.drop_duplicates()


Duplicados exactos: 0


In [6]:
nulls = df.isna().mean().sort_values(ascending=False)
nulls = (nulls * 100).round(2)
nulls[nulls > 0]


location            32.65
payment_method      25.79
item                 3.33
price_per_unit       1.79
total_spent          1.73
transaction_date     1.59
quantity             1.38
dtype: float64

In [7]:
invalid_values = ["ERROR", "UNKNOWN", "None", ""]

df = df.replace(invalid_values, np.nan)


In [8]:
def to_number(series):
    """Convierte texto a número. El punto es el separador decimal del dataset;
    los valores no numéricos (p. ej. "ERROR") quedan como NaN."""
    return pd.to_numeric(series.astype(str).str.strip(), errors="coerce")


In [9]:
num_cols = ["quantity", "price_per_unit", "total_spent"]

for c in num_cols:
    df[c] = to_number(df[c])


In [10]:
# Las fechas del dataset vienen en formato ISO (AAAA-MM-DD). Se indica el formato
# de forma explícita para que no se intercambien día y mes.
df["transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    format="%Y-%m-%d",
    errors="coerce"
)


In [11]:
df.dtypes
df.isna().sum()


transaction_id         0
item                 969
quantity             479
price_per_unit       533
total_spent          502
payment_method      3178
location            3961
transaction_date     460
dtype: int64

In [12]:
cat_cols = ["item", "payment_method", "location"]

for c in cat_cols:
    df[c] = df[c].fillna("Unknown")


In [13]:
df["quantity"] = df["quantity"].fillna(df["quantity"].median())
df["price_per_unit"] = df["price_per_unit"].fillna(df["price_per_unit"].median())


In [14]:
df["total_spent"] = df["quantity"] * df["price_per_unit"]


In [15]:
df = df.dropna(subset=["transaction_date"])


In [16]:
df["year"] = df["transaction_date"].dt.year
df["month"] = df["transaction_date"].dt.month
df["day_of_week"] = df["transaction_date"].dt.dayofweek


In [17]:
df.isna().sum()
df.head()
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 9540 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    9540 non-null   object        
 1   item              9540 non-null   object        
 2   quantity          9540 non-null   float64       
 3   price_per_unit    9540 non-null   float64       
 4   total_spent       9540 non-null   float64       
 5   payment_method    9540 non-null   object        
 6   location          9540 non-null   object        
 7   transaction_date  9540 non-null   datetime64[ns]
 8   year              9540 non-null   int32         
 9   month             9540 non-null   int32         
 10  day_of_week       9540 non-null   int32         
dtypes: datetime64[ns](1), float64(3), int32(3), object(4)
memory usage: 782.6+ KB


In [18]:
df.to_csv("../data/cleaned_cafe_sales.csv", index=False)
